In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/consolidated-pipeline/1st_setup/Utilities

In [0]:
print(gold_schema)

In [0]:
dbutils.widgets.text("Catalog","fmcg","catalog")
dbutils.widgets.text("Data_Source","orders","data_source")

catalog = dbutils.widgets.get("Catalog")
data_source = dbutils.widgets.get("Data_Source")

base_path = f"s3://amzn-s3-sportsbar/{data_source}"
landing_path = f"{base_path}/landing/"
processed_path = f"{base_path}/processed/"

print(f"Base Path : {base_path}")
print(f"Landing Path :{landing_path}")
print(f"Processed Path : {processed_path}")

bronze_table = f"{catalog}.{bronze_schema}.{data_source}"
silver_table = f"{catalog}.{silver_schema}.{data_source}"
gold_table = f"{catalog}.{gold_schema}.sb_fact_{data_source}"


# Bronze 

In [0]:
df = (
  spark.read
  .option("header", True)
  .option("inferSchema",True)
  .csv(f"{landing_path}/*.csv")
  .withColumn("readTimeStamp",F.current_timestamp())
  .select("*","_metadata.file_name","_metadata.file_size")
)
print("Total Rows:" ,df.count())
display(df.limit(5))

In [0]:
df =df.withColumn("product_id", F.col("product_id").cast("string")).withColumn("order_qty", F.col("order_qty").cast("string"))
df.printSchema()

In [0]:
df.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed",True)\
    .mode("append")\
    .saveAsTable(bronze_table)

In [0]:
51810+350

Staging table to process just the arrived incremental data


In [0]:
df.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", True)\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze_schema}.staging_{data_source}")

Moving files from source to processed directory in S3

In [0]:
files = dbutils.fs.ls(landing_path)
for file_info in files :
    dbutils.fs.mv(
        file_info.path,
        f"{processed_path}/{file_info.name}",
        True
    )

Silver Processing

In [0]:
df_orders = spark.sql(f"Select * from {catalog}.{bronze_schema}.staging_{data_source};")


In [0]:
display(df_orders.limit(350))

In [0]:
df_orders.count()

Transformations

In [0]:
# 1. Keep only rows where order_qty is present
df_orders = df_orders.filter(F.col("order_qty").isNotNull())

# 2. Clean customer_id -> keep numeric, else set to 999999
df_orders = df_orders.withColumn(
    "customer_id",
    F.when(F.col("customer_id").rlike("^[0-9]+$"), F.col("customer_id")).otherwise("999999")
)
#3.Remove weekday name from the date text
#  "Tuesday, July 01, 2025" → "July 01, 2025"

df_orders = df_orders.withColumn(
    "order_placement_date",
    F.regexp_replace(
        F.col("order_placement_date"), r"^[A-Za-z]+,\s*", ""
    )
)
# 4. Parse orde_placement_date using multiple possible formats
df_orders = df_orders.withColumn(
    "order_placement_date",
    F.coalesce(
        F.try_to_date("order_placement_date","yyyy/MM/dd"),
        F.try_to_date("order_placement_date","dd-MM-yyyy"),
        F.try_to_date("order_placement_date","dd/MM/yyyy"),
        F.try_to_date("order_placement_date","MMMM dd, yyyy"),
    )
)
# 5. Drop duplicates
df_orders = df_orders.dropDuplicates(
    ["order_id","order_placement_date","customer_id","product_id","order_qty"]
)

In [0]:
df_orders.count()

In [0]:
df_orders.select("order_placement_date").distinct().show(20, False)

In [0]:
display(df_orders.limit(10))

In [0]:
df_orders.agg(
    F.min("order_placement_date").alias("min_date"),
    F.max("order_placement_date").alias("max_date")
).show()

Join with products

In [0]:
df_products = spark.table("fmcg.silver.products")
df_joined = df_orders.join(
    df_products,
    on="product_id",
    how="inner"
).select(df_orders["*"],df_products["product_code"])

df_joined.show(5)

In [0]:
df_orders.count()

In [0]:
if not(spark.catalog.tableExists(silver_table)):
    df_joined.write.format("delta").option(
        "delta.enableChangeDataFeed",True
    ).option("mergeSchema",True).mode("overwrite").saveAsTable(silver_table)
else :
    silver_delta = DeltaTable.forName(spark,silver_table)
    silver_delta.alias("silver").merge(df_joined.alias("bronze"), "silver.order_placement_date = bronze.order_placement_date AND silver.order_id = bronze.order_id AND silver.product_code = bronze.product_code AND silver.customer_id = bronze.customer_id").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

Staging table to process arrived incremental data

In [0]:
df_joined.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed",True)\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{silver_schema}.staging_{data_source}")

#Gold

In [0]:
df_gold = spark.sql(f'Select order_id, order_placement_date as date, customer_id as customer_code,product_code,product_id, order_qty as sold_quantity from {catalog}.{silver_schema}.staging_{data_source};')
df_gold.show(5);

In [0]:
df_gold.count()

In [0]:
if not (spark.catalog.tableExists(gold_table)):
    print("creating new table")
    df_gold.write.format("delta").option(
        "delta.enableChangeDataFeed", True
    ).option("mergeSchema",True).mode("overwrite").saveAsTable(gold_table)
else :
    gold_delta=DeltaTable.forName(spark,gold_table)
    gold_delta.alias("source").merge(df_gold.alias("gold"),"source.date = gold.date AND source.order_id=gold.order_id AND source.product_code =gold.product_code AND source.customer_code = gold.customer_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

#Merging with Parent company

Note : We want data for monthly level but child data is on daily level

In [0]:
40811-41072


In [0]:
# df_child is for incremental daily rows
df_child = spark.sql(f'Select order_placement_date as date from {catalog}.{silver_schema}.staging_{data_source}')

incremental_month_df = df_child.select(
    F.trunc("date","MM").alias("start_month")
).distinct()

incremental_month_df.show()
incremental_month_df.createOrReplaceTempView("incremental_months")

In [0]:
monthly_table = spark.sql(f"""
                          Select date,product_code,customer_code, sold_quantity from {catalog}.{gold_schema}.sb_fact_orders sbf INNER JOIN incremental_months m ON trunc(sbf.date,'MM')=m.start_month """)
print("Total Rows:" ,monthly_table.count())
monthly_table.show(10)

In [0]:
monthly_table.select('date').distinct().orderBy('date').show()

In [0]:
df_monthly_recalc = (
    monthly_table
    .withColumn("month_start", F.trunc("date","MM"))
    .groupBy("month_start","product_code","customer_code")
    .agg(F.sum("sold_quantity").alias("sold_quantity"))
    .withColumnRenamed("month_start","date")
)
df_monthly_recalc.show(10,truncate=False)

In [0]:
df_monthly_recalc.count()

In [0]:
gold_parent_delta = DeltaTable.forName(spark, f"{catalog}.{gold_schema}.fact_orders")
gold_parent_delta.alias("parent_gold").merge(df_monthly_recalc.alias("child_gold"),"parent_gold.date = child_gold.date AND parent_gold.product_code = child_gold.product_code AND parent_gold.customer_code = child_gold.customer_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

#CleanUp

In [0]:
%sql
DROP TABLE fmcg.bronze.staging_orders;

In [0]:
%sql
DROP TABLE fmcg.silver.staging_orders;